In [7]:
import torch
import torch.nn as nn
from transformers import BertTokenizer, BertModel

# -------------------------------------------------
# Load pretrained BERT
# -------------------------------------------------

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
bert = BertModel.from_pretrained("bert-base-uncased")


# -------------------------------------------------
# BERT Classifier: entailment / similarity
# -------------------------------------------------

class BERTClassifier(nn.Module):
    def __init__(self, bert, num_labels):
        super().__init__()
        self.bert = bert
        self.classifier = nn.Linear(bert.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask, token_type_ids):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )

        cls_hidden = outputs.last_hidden_state[:, 0, :]  # [CLS]
        logits = self.classifier(cls_hidden)

        return logits


# -------------------------------------------------
# BERT Multiple Choice: QA / commonsense
# -------------------------------------------------

class BERTMultipleChoice(nn.Module):
    def __init__(self, bert):
        super().__init__()
        self.bert = bert
        self.scorer = nn.Linear(bert.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask, token_type_ids):
        batch_size, num_choices, seq_len = input_ids.shape

        input_ids = input_ids.view(batch_size * num_choices, seq_len)
        attention_mask = attention_mask.view(batch_size * num_choices, seq_len)
        token_type_ids = token_type_ids.view(batch_size * num_choices, seq_len)

        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )

        cls_hidden = outputs.last_hidden_state[:, 0, :]  # [CLS]
        scores = self.scorer(cls_hidden)

        return scores.view(batch_size, num_choices)


# -------------------------------------------------
# Helper tokenizer
# -------------------------------------------------

def tok_pair(text1, text2):
    return tokenizer(
        text1,
        text2,
        padding=True,
        truncation=True,
        return_tensors="pt"
    )


def tok_choices(context_question, answers):
    text1_list = [context_question] * len(answers)
    text2_list = answers

    batch = tokenizer(
        text1_list,
        text2_list,
        padding=True,
        truncation=True,
        return_tensors="pt"
    )

    input_ids = batch["input_ids"].unsqueeze(0)
    attention_mask = batch["attention_mask"].unsqueeze(0)
    token_type_ids = batch["token_type_ids"].unsqueeze(0)

    return input_ids, attention_mask, token_type_ids


# -------------------------------------------------
# Training functions
# -------------------------------------------------

def train_classifier(model, data, formatter, epochs=5):
    optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)
    loss_fn = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        total_loss = 0

        for item in data:
            text1, text2 = formatter(item)

            batch = tok_pair(text1, text2)
            label = torch.tensor([item["label"]])

            logits = model(
                batch["input_ids"],
                batch["attention_mask"],
                batch["token_type_ids"]
            )

            loss = loss_fn(logits, label)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch+1}, Loss={total_loss:.4f}")


def train_multiple_choice(model, data, formatter, epochs=5):
    optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)
    loss_fn = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        total_loss = 0

        for item in data:
            context_question, answers = formatter(item)

            input_ids, attention_mask, token_type_ids = tok_choices(
                context_question,
                answers
            )

            label = torch.tensor([item["label"]])

            scores = model(
                input_ids,
                attention_mask,
                token_type_ids
            )

            loss = loss_fn(scores, label)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch+1}, Loss={total_loss:.4f}")

In [8]:
# -------------------------------------------------
# 1. Entailment
# -------------------------------------------------

entailment_data = [
    {"premise": "A dog is running in the park.", "hypothesis": "An animal is outdoors.", "label": 0},
    {"premise": "The man is sleeping.", "hypothesis": "The man is running.", "label": 1},
    {"premise": "A woman is reading a book.", "hypothesis": "The woman is a teacher.", "label": 2},
]

entailment_labels = {
    0: "entailment",
    1: "contradiction",
    2: "neutral"
}

def format_entailment(item):
    return item["premise"], item["hypothesis"]

entail_model = BERTClassifier(bert, num_labels=3)
train_classifier(entail_model, entailment_data, format_entailment)

Epoch 1, Loss=4.0816
Epoch 2, Loss=1.9683
Epoch 3, Loss=1.0341
Epoch 4, Loss=0.4375
Epoch 5, Loss=0.2694


In [15]:
# -------------------------------------------------
# Prediction for Entailment
# -------------------------------------------------

entail_model.eval()

premise, hypothesis = format_entailment(
    entailment_data[0]
)

batch = tokenizer(
    premise,
    hypothesis,
    padding=True,
    truncation=True,
    return_tensors="pt"
)

with torch.no_grad():

    logits = entail_model(
        batch["input_ids"],
        batch["attention_mask"],
        batch["token_type_ids"]
    )

prediction = logits.argmax(dim=-1).item()

print("Premise:")
print(premise)

print("\nHypothesis:")
print(hypothesis)

print("\nPredicted:")
print(entailment_labels[prediction])

print("\nCorrect:")
print(
    entailment_labels[
        entailment_data[0]["label"]
    ]
)

Premise:
A dog is running in the park.

Hypothesis:
An animal is outdoors.

Predicted:
entailment

Correct:
entailment


In [9]:
# -------------------------------------------------
# 2. Similarity
# -------------------------------------------------

similarity_data = [
    {"text1": "A man is playing guitar.", "text2": "A person is making music.", "label": 1},
    {"text1": "A cat is sleeping.", "text2": "A car is driving.", "label": 0},
]

def format_similarity(item):
    return item["text1"], item["text2"]

sim_model = BERTClassifier(bert, num_labels=2)
train_classifier(sim_model, similarity_data, format_similarity)

Epoch 1, Loss=1.9653
Epoch 2, Loss=0.7513
Epoch 3, Loss=0.4402
Epoch 4, Loss=0.2307
Epoch 5, Loss=0.1171


In [14]:
# -------------------------------------------------
# Prediction for Similarity
# -------------------------------------------------

sim_model.eval()

text1, text2 = format_similarity(
    similarity_data[0]
)

batch = tokenizer(
    text1,
    text2,
    padding=True,
    truncation=True,
    return_tensors="pt"
)

with torch.no_grad():

    logits = sim_model(
        batch["input_ids"],
        batch["attention_mask"],
        batch["token_type_ids"]
    )

prediction = logits.argmax(dim=-1).item()

print("Text 1:")
print(text1)

print("\nText 2:")
print(text2)

print("\nPredicted:")
print(
    "similar"
    if prediction == 1
    else "not similar"
)

print("\nCorrect:")
print(
    "similar"
    if similarity_data[0]["label"] == 1
    else "not similar"
)

Text 1:
A man is playing guitar.

Text 2:
A person is making music.

Predicted:
similar

Correct:
similar


In [10]:
# -------------------------------------------------
# 3. Question Answering
# -------------------------------------------------

qa_data = [
    {
        "context": "The sky is clear and blue.",
        "question": "What color is the sky?",
        "answers": ["green", "blue", "red"],
        "label": 1
    }
]

def format_qa(item):
    context_question = item["context"] + " " + item["question"]
    return context_question, item["answers"]

qa_model = BERTMultipleChoice(bert)
train_multiple_choice(qa_model, qa_data, format_qa)

Epoch 1, Loss=1.0643
Epoch 2, Loss=0.6560
Epoch 3, Loss=0.3012
Epoch 4, Loss=0.0968
Epoch 5, Loss=0.0436


In [13]:
# -------------------------------------------------
# Prediction
# -------------------------------------------------

common_model.eval()

context_question, answers = format_qa(
    qa_data[0]
)

input_ids, attention_mask, token_type_ids = tok_choices(
    context_question,
    answers
)

with torch.no_grad():

    scores = common_model(
        input_ids,
        attention_mask,
        token_type_ids
    )

predicted_choice = scores.argmax(dim=-1).item()

print("Question:")
print(context_question)

print("\nChoices:")
for i, ans in enumerate(answers):
    print(i, ans)

print("\nPredicted Answer:")
print(answers[predicted_choice])

print("\nCorrect Answer:")
print(
    answers[
        commonsense_data[0]["label"]
    ]
)

Question:
The sky is clear and blue. What color is the sky?

Choices:
0 green
1 blue
2 red

Predicted Answer:
blue

Correct Answer:
green


In [11]:
# -------------------------------------------------
# 4. Commonsense Reasoning
# -------------------------------------------------

commonsense_data = [
    {
        "context": "John put ice cream in the sun.",
        "question": "What happened next?",
        "answers": [
            "It melted.",
            "It became colder.",
            "It turned into a rock."
        ],
        "label": 0
    }
]

def format_commonsense(item):
    context_question = item["context"] + " " + item["question"]
    return context_question, item["answers"]

common_model = BERTMultipleChoice(bert)
train_multiple_choice(common_model, commonsense_data, format_commonsense)

Epoch 1, Loss=1.0621
Epoch 2, Loss=0.7610
Epoch 3, Loss=0.4300
Epoch 4, Loss=0.2438
Epoch 5, Loss=0.1445


In [12]:
# -------------------------------------------------
# Prediction
# -------------------------------------------------

common_model.eval()

context_question, answers = format_commonsense(
    commonsense_data[0]
)

input_ids, attention_mask, token_type_ids = tok_choices(
    context_question,
    answers
)

with torch.no_grad():

    scores = common_model(
        input_ids,
        attention_mask,
        token_type_ids
    )

predicted_choice = scores.argmax(dim=-1).item()

print("Question:")
print(context_question)

print("\nChoices:")
for i, ans in enumerate(answers):
    print(i, ans)

print("\nPredicted Answer:")
print(answers[predicted_choice])

print("\nCorrect Answer:")
print(
    answers[
        commonsense_data[0]["label"]
    ]
)

Question:
John put ice cream in the sun. What happened next?

Choices:
0 It melted.
1 It became colder.
2 It turned into a rock.

Predicted Answer:
It melted.

Correct Answer:
It melted.
